# Multimodal Cancer Classification Challenge 2026 — v45 (stripped + noisy student)

**v45 = v44 minus the parts that hurt, plus a stronger pseudo-label teacher.**

What changes vs v44:

| Component | v44 | **v45 (this)** |
|---|---|---|
| Pseudo-label teacher | v41 (LB 0.7563) | **v44 (LB 0.7812)** — noisy-student iteration (Xie et al. 2020) |
| `USE_FL_TUNED_AUG` (CJ 0.5/0.3 + RandomGamma) | `True` | **`False`** — reverted to v41's BF-matched aug |
| `WEIGHT_DECAY` | `3e-4` | **`1e-4`** — reverted to v41's value |

Everything else identical to v44: 3 seeds × SWA(last 4 ep) + label smoothing 0.05 + dropout 0.4 + paired RandomResizedCrop + 40-way TTA (5 scales × 8 D4) + MIL + AdaBN + test stain norm.

## Why these specific changes

v43 (v44's full stack minus pseudo) landed at **0.7444** — a −0.012 regression from v41. So the FL-tuned aug + WD bump were **net-negative on their own**. v44 (0.7812) was carried entirely by pseudo-labels, which had to overcome v43's negative drag. v45 keeps the proven winner (pseudo + seed ensemble + SWA), strips the suspect parts, and upgrades the pseudo teacher to v44 itself.

The noisy-student iteration is well-grounded — Xie et al. 2020 (SOTA ImageNet 2019/20) showed that a single iteration with a stronger teacher reliably gains a few F1/AUC points when (a) the teacher has improved meaningfully over the previous round, and (b) the threshold filter rejects noisy labels. Both conditions hold here.

## Required Kaggle inputs

Attach both to v45 on Kaggle:
1. `rafaelproena/a3-adl` (competition data)
2. **`rafaelproena/submissionv44`** — **upload v44's `submission.csv` as a new private Kaggle dataset named `submissionv44`** (mirrors how you set up `submissionv41` for v44).

The config cell tries these paths in order until it finds one:

```
/kaggle/input/datasets/rafaelproena/submissionv44/submission.csv
/kaggle/input/submissionv44/submission.csv
/kaggle/input/v44-predictions/submission.csv
```

If none exist, cell 5 prints `WARNING: pseudo-label CSV not found at ...` and falls back to no-pseudo training. From v43 we know that fallback lands near 0.76, so **make sure the dataset is attached** before kicking off the 5-hour run.

## Expected lift

| Run | LB |
|---|---|
| v41 | 0.7563 |
| v43 (no pseudo, full v44 stack minus pseudo) | 0.7444 |
| **v44 (v43 + v41 pseudo)** | **0.7812** ← current best |
| v45 (stripped + v44 pseudo, this) | **~0.79–0.80 expected** |
| LB leader (current) | 0.7832 |

Gap to leader is just 0.0020 — even a marginal improvement plausibly tops. P(v45 > 0.7832 standalone) ≈ 55–65%.

## Compute on T4

Same shape as v44: ~5h with pseudo cells in the training set (~325s/epoch × 12 epochs × 3 seeds + AdaBN + 40-way TTA). SWA needs an extra BN-refresh pass per seed.

## Sanity check before the 5-hour commit

When cell 5 finishes, the log should show roughly:

```
Pseudo-labels loaded from /kaggle/input/datasets/rafaelproena/submissionv44/submission.csv
  threshold: <0.05 (neg) | >0.95 (pos)
  kept ~10000-12000 cells (v44 is sharper than v41, so more cells cross the thresholds)
  pseudo pos rate: ~0.58
  Combined training set: ~124000 cells
```

If you see `WARNING: pseudo-label CSV not found`, fix the dataset attachment **before** letting the run go.

## Suggested submission plan

You have 1 daily slot left today after v43/v44/v45_probe. Run v45 tonight, submit `submission.csv` (the 3-seed ensemble) tomorrow morning. If v45 > v44, swap final picks to {v45, v41}. If v45 ≤ v44, keep {v44, v41}.

Do **not** submit per-seed CSVs unless v45 ≥ 0.785 — within-recipe seed variance probes are only worth the slot at that point.


In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim.swa_utils as swa_utils
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/rafaelproena/a3-adl"),
    Path("/kaggle/input/a3-adl"),
    Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if (p / "train.csv").exists()), None)
assert DATA_ROOT is not None, f"train.csv not found at any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/kaggle/working")

# === v19 structural choices (kept) ===
USE_EFFICIENTNET    = True
USE_MIL_LOSS        = True
MIL_WEIGHT          = 0.5
USE_STRONG_AUG      = True
RANDOM_ERASING_P    = 0.25

USE_TEST_STAIN_NORM = True
USE_ADABN           = True

# === v41 changes (kept) ===
LABEL_SMOOTHING     = 0.05
DROPOUT             = 0.4
USE_PAIRED_CROP     = True
PAIRED_CROP_SCALE   = (0.85, 1.0)

# === v43-introduced features (kept from v44) ===
SEEDS               = [1, 2, 3]                  # multi-seed ensemble
SWA_EPOCHS          = 4                          # SWA over last 4 epochs of each seed
USE_EXTENDED_TTA    = True
TTA_SCALES          = (96, 112, 128, 144, 160)   # 5 scales x 8 D4 = 40-way

# === v45 STRIPPED: revert v43's FL-tuned aug + WD bump (suspect from v43 LB 0.7444) ===
# v43 = v41 + (these 4 changes: seeds+SWA, FL aug, WD bump, extended TTA) without
# pseudo regressed by -0.012 LB. seeds+SWA and extended TTA are well-validated
# regularizers and almost certainly NOT the culprit; FL aug and WD bump are.
# v45 reverts the suspect two to v41's settings while keeping pseudo + ensemble.
USE_FL_TUNED_AUG    = False                      # v44 was True; v45 REVERTS to v41
FL_COLORJITTER_B    = 0.5
FL_COLORJITTER_C    = 0.3
FL_GAMMA_LO         = 0.85
FL_GAMMA_HI         = 1.15
FL_GAMMA_P          = 0.5

# === v45: pseudo-labels from v44 (noisy student iteration, Xie et al. 2020) ===
# v44 (LB 0.7812) is a strictly stronger teacher than v41 (LB 0.7563). The same
# 0.05/0.95 confidence threshold should yield more cells AND fewer wrong labels.
# Upload v44's submission.csv as 'submissionv44' Kaggle dataset and attach.
USE_PSEUDO_LABELS    = True
_PSEUDO_LABEL_CANDIDATES = [
    "/kaggle/input/datasets/rafaelproena/submissionv44/submission.csv",
    "/kaggle/input/submissionv44/submission.csv",
    "/kaggle/input/v44-predictions/submission.csv",
]
PSEUDO_LABEL_CSV = next(
    (p for p in _PSEUDO_LABEL_CANDIDATES if Path(p).exists()),
    _PSEUDO_LABEL_CANDIDATES[0],  # default to first; load step will warn if missing
)
PSEUDO_THRESH_LOW    = 0.05
PSEUDO_THRESH_HIGH   = 0.95

# === Training (mostly v41) ===
EPOCHS      = 12
BATCH_SIZE  = 128
LR          = 3e-4
WEIGHT_DECAY = 1e-4    # v45: REVERTED from v44's 3e-4 to v41's 1e-4
GRAD_CLIP   = 1.0
MIXUP_ALPHA = 0.0

NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(SEEDS[0])

print(f"\nConfig (v45 - stripped + noisy student from v44):")
print(f"  SEEDS              = {SEEDS}  (multi-seed ensemble)")
print(f"  SWA_EPOCHS         = {SWA_EPOCHS}  (average last {SWA_EPOCHS} epochs per seed)")
print(f"  USE_FL_TUNED_AUG   = {USE_FL_TUNED_AUG}  (v44 True, REVERTED in v45)")
print(f"  WEIGHT_DECAY       = {WEIGHT_DECAY}  (v44 was 3e-4, REVERTED in v45)")
print(f"  USE_PSEUDO_LABELS  = {USE_PSEUDO_LABELS}  thresholds=({PSEUDO_THRESH_LOW}, {PSEUDO_THRESH_HIGH})")
print(f"  PSEUDO_LABEL_CSV   = {PSEUDO_LABEL_CSV}  (v45 uses v44 source; v44 used v41)")
print(f"  TTA_SCALES         = {TTA_SCALES}  ({8 * len(TTA_SCALES)}-way TTA)")
print(f"  LABEL_SMOOTHING    = {LABEL_SMOOTHING}  DROPOUT = {DROPOUT}  USE_PAIRED_CROP = {USE_PAIRED_CROP}")
print(f"  EPOCHS             = {EPOCHS}  BATCH_SIZE = {BATCH_SIZE}")


In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    """v43: caches passed in are pre-merged (train + pseudo-from-test)."""
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
        self.has_pid = "patient_id" in self.df.columns
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        pid = int(row["patient_id"]) if self.has_pid else -1
        return {"bf": bf, "fl": fl, "label": label, "patient_id": pid, "name": name}

class PatientBalancedSampler(Sampler):
    """Each batch contains cells from `patients_per_batch` distinct patients.

    v43: patient_id=-1 (pseudo cells) is treated as one extra 'patient' group; the
    sampler then draws from {12 real patients + 1 pseudo group} with equal probability.
    Pseudo cells appear in ~30% of batches at ~32 cells/batch.
    """
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd

def _make_effnet_b0_branch(pretrained=True):
    import timm
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                            num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                         stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280

def _make_branch(pretrained=True):
    if USE_EFFICIENTNET:
        return _make_effnet_b0_branch(pretrained)
    return _make_resnet18_branch(pretrained)

class MultimodalClassifier(nn.Module):
    """v19/v41 architecture: dual EffNet-B0 branches + late concat fusion head."""
    def __init__(self, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.bf_branch, fd = _make_branch(pretrained)
        self.fl_branch, _  = _make_branch(pretrained)
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

with torch.no_grad():
    _m = MultimodalClassifier(pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    n_params = sum(p.numel() for p in _m.parameters())
    print(f"Output shape: {_m(_x, _x).shape}   params: {n_params / 1e6:.1f}M")
    print(f"Backbone: {'EfficientNet-B0' if USE_EFFICIENTNET else 'ResNet-18'}")
    del _m, _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")

# Stain stats (used to overwrite BF_MEAN/STD etc below; same scheme as v41)
def _sample_pixel_stats(cache, names, n_sample=1500):
    rng = np.random.default_rng(42)
    sampled = rng.choice(names, size=min(n_sample, len(names)), replace=False)
    pixels = []
    for n in sampled:
        img = np.asarray(Image.open(io.BytesIO(cache[n])).convert("L"),
                         dtype=np.float32) / 255.0
        pixels.append(img.ravel())
    pixels = np.concatenate(pixels)
    return float(pixels.mean()), float(pixels.std())

if USE_TEST_STAIN_NORM:
    BF_MEAN_T, BF_STD_T = _sample_pixel_stats(bf_test_cache, df_test["Name"].tolist())
    FL_MEAN_T, FL_STD_T = _sample_pixel_stats(fl_test_cache, df_test["Name"].tolist())
    BF_MEAN_R, BF_STD_R = _sample_pixel_stats(bf_train_cache, df_train["Name"].tolist())
    FL_MEAN_R, FL_STD_R = _sample_pixel_stats(fl_train_cache, df_train["Name"].tolist())
    print(f"\nPixel statistics:")
    print(f"  BF train: mean={BF_MEAN_R:.4f} std={BF_STD_R:.4f}")
    print(f"  BF test:  mean={BF_MEAN_T:.4f} std={BF_STD_T:.4f}")
    print(f"  FL train: mean={FL_MEAN_R:.4f} std={FL_STD_R:.4f}")
    print(f"  FL test:  mean={FL_MEAN_T:.4f} std={FL_STD_T:.4f}")
    BF_MEAN, BF_STD = BF_MEAN_T, BF_STD_T
    FL_MEAN, FL_STD = FL_MEAN_T, FL_STD_T
    print(f"  -> using TEST stats")

# === v45: load pseudo-labels from v44 (noisy student iteration) ===
df_pseudo = None
combined_bf_cache = bf_train_cache
combined_fl_cache = fl_train_cache
df_train_combined = df_train

if USE_PSEUDO_LABELS:
    if Path(PSEUDO_LABEL_CSV).exists():
        df_pseudo_raw = pd.read_csv(PSEUDO_LABEL_CSV)
        confident_mask = ((df_pseudo_raw["Diagnosis"] < PSEUDO_THRESH_LOW) |
                          (df_pseudo_raw["Diagnosis"] > PSEUDO_THRESH_HIGH))
        df_pseudo = df_pseudo_raw[confident_mask].copy()
        df_pseudo["Diagnosis"] = (df_pseudo["Diagnosis"] > 0.5).astype(int)
        df_pseudo["patient_id"] = -1
        n_pseudo = len(df_pseudo)
        n_pos = int((df_pseudo["Diagnosis"] == 1).sum())
        n_neg = n_pseudo - n_pos
        print(f"\nPseudo-labels loaded from {PSEUDO_LABEL_CSV}")
        print(f"  threshold: <{PSEUDO_THRESH_LOW} (neg) | >{PSEUDO_THRESH_HIGH} (pos)")
        print(f"  kept {n_pseudo} cells ({n_pos} pos, {n_neg} neg) of {len(df_pseudo_raw)} total")
        print(f"  pseudo pos rate: {n_pos/max(n_pseudo,1):.4f} (train was {df_train['Diagnosis'].mean():.4f})")

        combined_bf_cache = {**bf_train_cache, **bf_test_cache}
        combined_fl_cache = {**fl_train_cache, **fl_test_cache}
        df_train_combined = pd.concat([df_train, df_pseudo[df_train.columns]], ignore_index=True)
        print(f"  Combined training set: {len(df_train_combined)} cells "
              f"({len(df_train)} real + {n_pseudo} pseudo)")
    else:
        print(f"\nWARNING: pseudo-label CSV not found at {PSEUDO_LABEL_CSV}")
        print(f"  Attach the 'submissionv44' Kaggle dataset to enable pseudo-labels.")
        print(f"  Falling back to training without pseudo-labels.")


In [ ]:
def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class RandomGamma:
    """Random gamma correction on PIL 'L' images; gamma drawn uniformly in [lo, hi]."""
    def __init__(self, lo=0.85, hi=1.15, p=0.5):
        self.lo, self.hi, self.p = lo, hi, p
    def __call__(self, img):
        if random.random() < self.p:
            g = random.uniform(self.lo, self.hi)
            img = TF.adjust_gamma(img, gamma=g)
        return img

class PairedGeoAug:
    """D4 + small rotation + paired affine + paired RandomResizedCrop (from v41)."""
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0,
                 affine_deg=0.0, affine_translate=0.0,
                 paired_crop=False, paired_crop_scale=(0.85, 1.0)):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
        self.affine_deg = affine_deg; self.affine_translate = affine_translate
        self.paired_crop = paired_crop
        self.paired_crop_scale = paired_crop_scale
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        if self.affine_deg > 0 or self.affine_translate > 0:
            H, W = bf.shape[-2], bf.shape[-1]
            angle = random.uniform(-self.affine_deg, self.affine_deg) if self.affine_deg > 0 else 0.0
            tx = random.uniform(-self.affine_translate, self.affine_translate) * W if self.affine_translate > 0 else 0
            ty = random.uniform(-self.affine_translate, self.affine_translate) * H if self.affine_translate > 0 else 0
            bf = TF.affine(bf, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
            fl = TF.affine(fl, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
        if self.paired_crop:
            H, W = bf.shape[-2], bf.shape[-1]
            scale = random.uniform(self.paired_crop_scale[0], self.paired_crop_scale[1])
            new_h = max(1, int(round(H * scale)))
            new_w = max(1, int(round(W * scale)))
            top  = random.randint(0, H - new_h) if H > new_h else 0
            left = random.randint(0, W - new_w) if W > new_w else 0
            bf = TF.resized_crop(bf, top, left, new_h, new_w, size=(H, W),
                                 interpolation=TF.InterpolationMode.BILINEAR, antialias=True)
            fl = TF.resized_crop(fl, top, left, new_h, new_w, size=(H, W),
                                 interpolation=TF.InterpolationMode.BILINEAR, antialias=True)
        return bf, fl

def train_modality_transform(modality):
    """v45: FL-tuned aug DISABLED (reverted to v41's BF-matched aug). The branch is
    still here so the variable USE_FL_TUNED_AUG can be flipped back True for ablation."""
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    if modality == "fl" and USE_FL_TUNED_AUG:
        steps = [
            T.ColorJitter(brightness=FL_COLORJITTER_B, contrast=FL_COLORJITTER_C),
            RandomGamma(FL_GAMMA_LO, FL_GAMMA_HI, p=FL_GAMMA_P),
            norm,
        ]
    elif USE_STRONG_AUG:
        steps = [T.ColorJitter(brightness=0.4, contrast=0.4), norm]
    else:
        steps = [T.ColorJitter(brightness=0.2, contrast=0.2), norm]
    if USE_STRONG_AUG and RANDOM_ERASING_P > 0:
        steps.append(T.RandomErasing(p=RANDOM_ERASING_P, scale=(0.02, 0.20),
                                     ratio=(0.3, 3.3), value=0.0))
    return T.Compose(steps)

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

def build_paired_aug():
    return PairedGeoAug(max_rot=10.0, affine_deg=15.0, affine_translate=0.10,
                        paired_crop=USE_PAIRED_CROP,
                        paired_crop_scale=PAIRED_CROP_SCALE)

print("Augmentation summary (v45):")
print(f"  BF:  ColorJitter(0.4, 0.4) + RandomErasing(p={RANDOM_ERASING_P}) [v41 / unchanged]")
if USE_FL_TUNED_AUG:
    print(f"  FL:  ColorJitter({FL_COLORJITTER_B}, {FL_COLORJITTER_C}) + RandomGamma({FL_GAMMA_LO}, {FL_GAMMA_HI}) p={FL_GAMMA_P}  [v43 FL-tuned, currently ENABLED]")
else:
    print(f"  FL:  ColorJitter(0.4, 0.4) + RandomErasing(p={RANDOM_ERASING_P}) [v41 BF-matched / v45 stripped]")
print(f"  Geom (paired): D4 + +-10 rot + +-15 affine + 10% translate + RandomResizedCrop scale={PAIRED_CROP_SCALE}")

In [ ]:
def smooth(y, eps):
    if eps <= 0: return y
    return y * (1.0 - eps) + eps * 0.5

def mil_patient_loss(logits, y, patient_ids, pos_weight=None):
    """Per-patient mean-logit BCE. v43+: skips pseudo cells (patient_id == -1)."""
    real_mask = patient_ids >= 0
    if real_mask.sum() < 2:
        return torch.zeros((), device=logits.device, dtype=logits.dtype)
    logits_r = logits[real_mask]
    y_r = y[real_mask]
    pids_r = patient_ids[real_mask]
    unique_pids = torch.unique(pids_r)
    if len(unique_pids) < 2:
        return torch.zeros((), device=logits.device, dtype=logits.dtype)
    p_logits = []
    p_labels = []
    for pid in unique_pids:
        mask = pids_r == pid
        p_logits.append(logits_r[mask].mean())
        p_labels.append(y_r[mask][0])
    p_logits = torch.stack(p_logits)
    p_labels = torch.stack(p_labels)
    return F.binary_cross_entropy_with_logits(p_logits, p_labels.float(),
                                              pos_weight=pos_weight)

def run_epoch_train(model, loader, optimizer, scaler, criterion_cell, sched,
                    pos_weight=None, log_every=200):
    model.train()
    losses, hard_ys, ps = [], [], []
    cell_losses, mil_losses = [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf  = batch["bf"].to(DEVICE, non_blocking=True)
        fl  = batch["fl"].to(DEVICE, non_blocking=True)
        y   = batch["label"].float().to(DEVICE, non_blocking=True)
        pid = batch["patient_id"].to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())
        y_s = smooth(y, LABEL_SMOOTHING)
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss_cell = criterion_cell(logits, y_s)
            if USE_MIL_LOSS:
                loss_mil = mil_patient_loss(logits, y, pid, pos_weight=pos_weight)
                loss = loss_cell + MIL_WEIGHT * loss_mil
            else:
                loss_mil = torch.zeros((), device=logits.device)
                loss = loss_cell
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            old_scale = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale() >= old_scale: sched.step()
        else:
            loss.backward()
            if GRAD_CLIP > 0: nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step(); sched.step()
        losses.append(loss.item())
        cell_losses.append(loss_cell.item())
        mil_losses.append(loss_mil.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | "
                  f"total {float(np.mean(losses[-log_every:])):.4f} "
                  f"cell {float(np.mean(cell_losses[-log_every:])):.4f} "
                  f"mil {float(np.mean(mil_losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), float(np.mean(cell_losses)), float(np.mean(mil_losses)), auc


@torch.no_grad()
def update_bn_two_inputs(loader, model, device=None):
    """BatchNorm-stats refresh for an AveragedModel with two-input forward.
    Reimplements torch.optim.swa_utils.update_bn for our (bf, fl) signature."""
    momenta = {}
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.reset_running_stats()
            momenta[module] = module.momentum
    if not momenta:
        return
    was_training = model.training
    model.train()
    n = 0
    for batch in loader:
        bf = batch["bf"].to(device, non_blocking=True)
        fl = batch["fl"].to(device, non_blocking=True)
        b = bf.size(0)
        mom = b / float(n + b)
        for module in momenta:
            module.momentum = mom
        model(bf, fl)
        n += b
    for module in momenta:
        module.momentum = momenta[module]
    model.train(was_training)


def train_one_seed(seed):
    """Train one model end-to-end with SWA on last SWA_EPOCHS epochs.
    Returns dict with seed, ckpt path, and history list."""
    print(f"\n{'='*70}\n=== v45: Training seed={seed} (with SWA last {SWA_EPOCHS} epochs) ===\n{'='*70}")
    seed_everything(seed + 100)

    train_ds = CachedCellDataset(df_train_combined, combined_bf_cache, combined_fl_cache,
                                 train_modality_transform("bf"),
                                 train_modality_transform("fl"),
                                 paired_tf=build_paired_aug())
    sampler = PatientBalancedSampler(df_train_combined, batch_size=BATCH_SIZE,
                                     patients_per_batch=PATIENTS_PER_BATCH, seed=seed + 100)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

    model = MultimodalClassifier(pretrained=True, dropout=DROPOUT).to(DEVICE)
    swa_model = swa_utils.AveragedModel(model)
    swa_start = EPOCHS - SWA_EPOCHS

    pos = (df_train_combined["Diagnosis"] == 1).sum()
    neg = (df_train_combined["Diagnosis"] == 0).sum()
    pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
    print(f"  pos_weight={pos_weight.item():.3f}  LR={LR}  WD={WEIGHT_DECAY}  "
          f"MIL_W={MIL_WEIGHT if USE_MIL_LOSS else 0.0}  seed={seed}  "
          f"SWA averages epochs [{swa_start}..{EPOCHS-1}]")
    criterion_cell = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, steps_per_epoch=len(train_loader),
        epochs=EPOCHS, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

    history = []
    for ep in range(EPOCHS):
        t0 = time.time()
        tr_loss, tr_cell, tr_mil, tr_auc = run_epoch_train(
            model, train_loader, optimizer, scaler, criterion_cell, sched,
            pos_weight=pos_weight)
        swa_active = ep >= swa_start
        if swa_active:
            swa_model.update_parameters(model)
        dt = time.time() - t0
        swa_tag = "[SWA]" if swa_active else "     "
        print(f"  [seed={seed}] ep {ep:>2d} {swa_tag} | total {tr_loss:.4f} cell {tr_cell:.4f} "
              f"mil {tr_mil:.4f} tr_auc {tr_auc:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_cell": tr_cell,
                        "tr_mil": tr_mil, "tr_auc": tr_auc, "time": dt,
                        "swa_active": swa_active})

    # SWA needs BN stats updated for the averaged weights.
    print(f"  [seed={seed}] running update_bn pass on training set for SWA model...")
    t0 = time.time()
    update_bn_two_inputs(train_loader, swa_model, device=DEVICE)
    print(f"    BN refresh done in {time.time()-t0:.1f}s")

    ckpt_path = OUT_DIR / f"swa_seed{seed}.pt"
    torch.save({"model": swa_model.module.state_dict(), "epoch": EPOCHS - 1,
                "args": {"dropout": DROPOUT,
                         "backbone": "efficientnet_b0" if USE_EFFICIENTNET else "resnet18",
                         "seed": seed, "swa": True, "swa_epochs": SWA_EPOCHS}},
               ckpt_path)
    with open(OUT_DIR / f"history_seed{seed}.json", "w") as f:
        json.dump({"seed": seed, "swa_epochs": SWA_EPOCHS, "history": history}, f, indent=2)
    print(f"  [seed={seed}] saved SWA ckpt: {ckpt_path}")

    del model, swa_model, optimizer, sched, scaler, train_loader, train_ds, sampler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {"seed": seed, "ckpt": str(ckpt_path), "history": history}

In [ ]:
trained_seeds = []
overall_t0 = time.time()
for seed in SEEDS:
    result = train_one_seed(seed)
    trained_seeds.append(result)
    elapsed_min = (time.time() - overall_t0) / 60.0
    print(f"  [v45] cumulative training wall-clock so far = {elapsed_min:.1f} min "
          f"({len(trained_seeds)}/{len(SEEDS)} seeds done)")

print(f"\n[v45] All {len(trained_seeds)} seeds trained.")
for t in trained_seeds:
    final_auc = t["history"][-1]["tr_auc"]
    print(f"  seed={t['seed']}  final tr_auc={final_auc:.4f}")

In [ ]:
# Per-seed learning curves overlaid (loss, AUC, time).
n_seeds = len(trained_seeds)
fig, axes = plt.subplots(n_seeds, 3, figsize=(13, 3.2 * n_seeds), squeeze=False)
for row, t in enumerate(trained_seeds):
    seed = t["seed"]; h = t["history"]
    epochs = [e["epoch"] for e in h]
    ax_l, ax_a, ax_t = axes[row]
    ax_l.plot(epochs, [e["tr_loss"] for e in h], marker="o", color="tab:blue", label="total")
    ax_l.plot(epochs, [e["tr_cell"] for e in h], marker="s", color="tab:purple", label="cell BCE")
    ax_l.plot(epochs, [e["tr_mil"]  for e in h], marker="^", color="tab:orange", label="MIL BCE")
    # Mark SWA epochs
    for e in h:
        if e["swa_active"]:
            ax_l.axvspan(e["epoch"] - 0.4, e["epoch"] + 0.4, color="tab:green", alpha=0.07)
            ax_a.axvspan(e["epoch"] - 0.4, e["epoch"] + 0.4, color="tab:green", alpha=0.07)
    ax_l.legend(); ax_l.set(title=f"[seed={seed}] Train losses", xlabel="epoch", ylabel="loss"); ax_l.grid(True)
    ax_a.plot(epochs, [e["tr_auc"]  for e in h], marker="o", color="tab:green")
    ax_a.set(title=f"[seed={seed}] Train AUC (cell)", xlabel="epoch", ylabel="AUC"); ax_a.grid(True)
    ax_t.plot(epochs, [e["time"]    for e in h], marker="o", color="tab:red")
    ax_t.set(title=f"[seed={seed}] Epoch time (s)",  xlabel="epoch", ylabel="seconds"); ax_t.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
@torch.no_grad()
def adabn_pass(model, loader):
    model.train()
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            _ = model(bf, fl)
    model.eval()

def _d4_at_scale(bf, fl, scale=None):
    if scale is not None and scale != bf.shape[-1]:
        bf = F.interpolate(bf, size=(scale, scale), mode="bilinear", align_corners=False)
        fl = F.interpolate(fl, size=(scale, scale), mode="bilinear", align_corners=False)
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def _multiscale_tta(bf, fl, scales):
    for s in scales:
        for bf_t, fl_t in _d4_at_scale(bf, fl, scale=s):
            yield bf_t, fl_t

def load_model(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    model = MultimodalClassifier(pretrained=False, dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    return model

def predict_one(ckpt_path, loader, tta_scales=None):
    model = load_model(ckpt_path)
    if USE_ADABN:
        print("  running AdaBN pass on test...")
        t0 = time.time()
        adabn_pass(model, loader)
        print(f"    AdaBN done in {time.time()-t0:.1f}s")
    aug_fn = (lambda bf, fl: _multiscale_tta(bf, fl, tta_scales)) if tta_scales \
             else (lambda bf, fl: _d4_at_scale(bf, fl))
    n_aug = 8 * (len(tta_scales) if tta_scales else 1)
    preds = []
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in aug_fn(bf, fl):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

# Shared test loader (use TEST cache only — no pseudo-label mixing at inference).
test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

scales_to_use = TTA_SCALES if USE_EXTENDED_TTA else (112, 128, 144)
n_aug_total = 8 * len(scales_to_use)
print(f"Predicting with {n_aug_total}-way TTA (scales={scales_to_use}, AdaBN={USE_ADABN})  "
      f"for {len(trained_seeds)} SWA seeds")

per_seed_preds = {}
for t in trained_seeds:
    seed = t["seed"]
    print(f"\n--- Predict [seed={seed}] ---")
    t0 = time.time()
    preds = predict_one(t["ckpt"], test_loader, tta_scales=scales_to_use)
    per_seed_preds[seed] = preds
    pm_path = WORK_DIR / f"submission_seed{seed}.csv"
    pd.DataFrame({"Name": df_test["Name"].values,
                  "Diagnosis": preds}).to_csv(pm_path, index=False)
    print(f"  [seed={seed}] done in {time.time()-t0:.1f}s  "
          f"mean={preds.mean():.4f}  <0.05={(preds<0.05).mean():.2%}  >0.95={(preds>0.95).mean():.2%}")

# Ensemble: sigmoid-average across all seeds (same-arch, same-recipe).
print(f"\n--- Building ensemble of {len(per_seed_preds)} seeds (sigmoid average) ---")
ensemble_preds = np.mean(list(per_seed_preds.values()), axis=0)
sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": ensemble_preds})
sub.to_csv(WORK_DIR / "submission.csv", index=False)
print(f"Auto-submitted ensemble (sigmoid avg of {len(per_seed_preds)} seeds): "
      f"mean={ensemble_preds.mean():.4f}  min={ensemble_preds.min():.4f}  max={ensemble_preds.max():.4f}  "
      f"<0.05={(ensemble_preds<0.05).mean():.2%}  >0.95={(ensemble_preds>0.95).mean():.2%}")
print(sub.head())

# Summary of all CSVs written.
print(f"\n=== Output summary ===")
for f in sorted(WORK_DIR.glob("submission*.csv")):
    arr = pd.read_csv(f)["Diagnosis"].values
    print(f"  {f.name:>35}  rows={len(arr):>6}  mean={arr.mean():.4f}  std={arr.std():.4f}")
!wc -l /kaggle/working/submission*.csv